# 02. 모델 학습 (Training)

전처리된 데이터로 LightGBM 회귀 모델을 학습합니다.

**학습 순서**
1. 전처리 데이터 로드
2. 특성(Feature) / 정답(Label) 분리
3. Train / Test Split (8:2)
4. LightGBM 학습 (v1 기본 → v2 시차 변수 포함 → v3 데이터 정제 후)
5. 성능 평가 (RMSE, R²)

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

from google.colab import drive
drive.mount('/content/drive')

# 전처리 완료 데이터 로드
df_final = pd.read_csv('/content/drive/MyDrive/df_final_preprocessed.csv', parse_dates=['date'])
print(f'데이터 로드 완료: {len(df_final)}개')
print(df_final.head())

## 1. 특성 / 정답 분리 및 데이터 분할

In [ ]:
FEATURES = ['temp', 'humidity', 'VPD', 'solar_rad', 'GDD',
            'temp_3d_avg', 'hum_3d_avg', 'solar_3d_avg', 'VPD_3d_avg']
TARGET   = 'weeklyGrowth'

X = df_final[FEATURES]
y = df_final[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'학습: {len(X_train)}개 / 테스트: {len(X_test)}개')

## 2. LightGBM 모델 학습 및 평가

In [ ]:
model = lgb.LGBMRegressor(n_estimators=200, learning_rate=0.05, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)

print('=== 모델 성능 ===')
print(f'RMSE : {rmse:.4f}  (낮을수록 좋음)')
print(f'R²   : {r2:.4f}  (1.0에 가까울수록 좋음)')

## 3. 모델 저장

In [ ]:
import joblib

model_save_path = '/content/drive/MyDrive/lgbm_tomato_growth_model.pkl'
joblib.dump(model, model_save_path)
print(f'모델 저장 완료: {model_save_path}')